### **ABOUT THE DATASET**
1. ID: Unique ID for a news article
2. TITLE: The title of a news article
3. TEXT: The text of the article; could be incomplete
4. AUTHOR: Authoer of the news article
5. LABEL_DATA: A lable_data marks whether the news is real or fake?
6. LABEL: A lable marks real == 0 or fake == 1

### **IMPORTING THE DEPENDENCIES**

In [1]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\aveng\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
# printing the stopwords in english
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

### **DATA PREPROCESSING**

In [4]:
# Loading the dataset to pandas dataframe
news_dataset = pd.read_csv('WELFake_Dataset.csv')

In [5]:
news_dataset.shape

(72134, 4)

In [6]:
# printing the first 5 rows of the dataset
news_dataset.head(5)

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [7]:
# containing the number of missing values in the dataset
news_dataset.isnull().sum()

Unnamed: 0      0
title         558
text           39
label           0
dtype: int64

In [8]:
# Drop the null values
news_dataset = news_dataset.dropna()

In [9]:
# # merging the title and news text
news_dataset['content'] = news_dataset['title']+' '+news_dataset['text']

In [10]:
print(news_dataset['content'])

0        LAW ENFORCEMENT ON HIGH ALERT Following Threat...
2        UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...
3        Bobby Jindal, raised Hindu, uses story of Chri...
4        SATAN 2: Russia unvelis an image of its terrif...
5        About Time! Christian Group Sues Amazon and SP...
                               ...                        
72129    Russians steal research on Trump in hack of U....
72130     WATCH: Giuliani Demands That Democrats Apolog...
72131    Migrants Refuse To Leave Train At Refugee Camp...
72132    Trump tussle gives unpopular Mexican leader mu...
72133    Goldman Sachs Endorses Hillary Clinton For Pre...
Name: content, Length: 71537, dtype: object


In [11]:
# separating the data & label
X = news_dataset.drop(columns = 'label', axis=1)
Y = news_dataset['label']

In [12]:
print(X)
print(Y)

       Unnamed: 0                                              title  \
0               0  LAW ENFORCEMENT ON HIGH ALERT Following Threat...   
2               2  UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...   
3               3  Bobby Jindal, raised Hindu, uses story of Chri...   
4               4  SATAN 2: Russia unvelis an image of its terrif...   
5               5  About Time! Christian Group Sues Amazon and SP...   
...           ...                                                ...   
72129       72129  Russians steal research on Trump in hack of U....   
72130       72130   WATCH: Giuliani Demands That Democrats Apolog...   
72131       72131  Migrants Refuse To Leave Train At Refugee Camp...   
72132       72132  Trump tussle gives unpopular Mexican leader mu...   
72133       72133  Goldman Sachs Endorses Hillary Clinton For Pre...   

                                                    text  \
0      No comment is expected from Barack Obama Membe...   
2       Now, mo

### **STEMMING**: Stemming is the process of reducing word to its Root word

In [13]:
port_stem = PorterStemmer()

In [14]:
def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]',' ',content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
    stemmed_content = ' '.join(stemmed_content)
    return stemmed_content

In [15]:
news_dataset['content'] = news_dataset['content'].apply(stemming)

In [16]:
print(news_dataset['content'])

0        law enforc high alert follow threat cop white ...
2        unbeliev obama attorney gener say charlott rio...
3        bobbi jindal rais hindu use stori christian co...
4        satan russia unv imag terrifi new supernuk wes...
5        time christian group sue amazon splc design ha...
                               ...                        
72129    russian steal research trump hack u democrat p...
72130    watch giuliani demand democrat apolog trump ra...
72131    migrant refus leav train refuge camp hungari m...
72132    trump tussl give unpopular mexican leader much...
72133    goldman sach endors hillari clinton presid gol...
Name: content, Length: 71537, dtype: object


In [17]:
# separating the data and label
X = news_dataset['content'].values
Y = news_dataset['label'].values

In [18]:
print(X)

['law enforc high alert follow threat cop white blacklivesmatt fyf terrorist video comment expect barack obama member fyf fukyoflag blacklivesmatt movement call lynch hang white peopl cop encourag other radio show tuesday night turn tide kill white peopl cop send messag kill black peopl america one f yoflag organ call sunshin radio blog show host texa call sunshin f ing opinion radio show snapshot fyf lolatwhitefear twitter page p show urg support call fyf tonight continu dismantl illus white snapshot twitter radio call invit fyf radio show air p eastern standard time show caller clearli call lynch kill white peopl minut clip radio show heard provid breitbart texa someon would like refer hannib alreadi receiv death threat result interrupt fyf confer call unidentifi black man said mother f ker start f ing like us bunch ni er takin one us roll said caus alreadi roll gang anyway six seven black mother f cker see white person lynch ass let turn tabl conspir cop start lose peopl state emerg

In [19]:
print(Y)

[1 1 0 ... 0 0 1]


In [20]:
Y.shape

(71537,)

In [21]:
X.shape

(71537,)

In [22]:
# converting the textual data into numerical data
vectorizer = TfidfVectorizer()
vectorizer.fit(X)

X = vectorizer.transform(X)

In [23]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 13640688 stored elements and shape (71537, 161943)>
  Coords	Values
  (0, 936)	0.019094182800048216
  (0, 1280)	0.017347111400182653
  (0, 2127)	0.0524876547542501
  (0, 2778)	0.020218810602040272
  (0, 3606)	0.02990572361868685
  (0, 3991)	0.027465665890226604
  (0, 4255)	0.02386029214171958
  (0, 4326)	0.05050914863050092
  (0, 4836)	0.015121083026304497
  (0, 4852)	0.024823750505703946
  (0, 6001)	0.014577564390257154
  (0, 6494)	0.05730010107068436
  (0, 6831)	0.015870818888082696
  (0, 8423)	0.12659846575719516
  (0, 8961)	0.01549779283694407
  (0, 10463)	0.06697964736530035
  (0, 11410)	0.018946331102593025
  (0, 12704)	0.015784839576180675
  (0, 14045)	0.01832762091811868
  (0, 14651)	0.01783750531931106
  (0, 15413)	0.19269510054097194
  (0, 15470)	0.08128065505543595
  (0, 15582)	0.08892769046455246
  (0, 15856)	0.029328301995583207
  (0, 18030)	0.10835521235789827
  :	:
  (71536, 132424)	0.031699711506764296
  (715

### **SPLITTING THE DATASET TO TRAINING AND TEST DATA**

In [24]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, stratify = Y, random_state = 2)

### Training the model: **LOGISTIC REGRESSION**

In [25]:
model = LogisticRegression()

In [26]:
model.fit(X_train, Y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


### EVALUATION: **Accuracy Score**

In [27]:
# accuracy score on the training data
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)

In [28]:
print('Accuracy score of the training data : ', training_data_accuracy)

Accuracy score of the training data :  0.9626937391881738


In [29]:
# accuracy score on the test data
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)

In [30]:
print('Accuracy score of the test data : ', test_data_accuracy)

Accuracy score of the test data :  0.9468828627341348


### **MAKING A PREDICTIVE SYSTEM**

In [31]:
X_new = X_test[158]

prediction = model.predict(X_new)
print(prediction)

if (prediction[0]==0):
    print('The news is real')
else:
    print('The news is fake')

[1]
The news is fake


In [32]:
import pickle

# 1. Save your trained machine learning model
with open('fake_news_model.pkl', 'wb') as model_file:
    pickle.dump(model, model_file)

# 2. Save your fitted TF-IDF Vectorizer
with open('tfidf_vectorizer.pkl', 'wb') as vectorizer_file:
    pickle.dump(vectorizer, vectorizer_file)

print("Success! Both files have been generated.")

Success! Both files have been generated.
